---
### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 1.4 — Adding Retrieval to the Agent
---
## 💬 Product Team Check-in

> **Sam:** "Hey - users are noticing right away the agent lacks basic knowledge about the latest features of MLflow 3.0 and above. Are we in too deep already or is there a way we can fix this?"

**Where we are:** Our MLflow assistant answers questions, but it struggles to gather context about the latest features of MLflow because they are outside the model's knowledge cutoff.

In this notebook we'll:
1. Build a simple document store with MLflow 3 knowledge
2. Create a RAG pipeline from scratch — embedding, search, prompt construction
3. **Then** add MLflow Tracing to see inside the pipeline

We intentionally build the pipeline *without* tracing first, then layer it on — so you can see exactly what tracing adds.

---
## 0 - Connect to MLflow

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

mlflow.openai.autolog()

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url=os.environ["GEMINI_OPENAI_BASE_URL"],
)

MODEL = "gemini-2.5-flash-lite"
JUDGE_MODEL = "gemini:/gemini-3.1-flash-lite-preview"
EMBEDDING_MODEL = "gemini-embedding-001"

EXPERIMENT_NAME = os.environ.get("EXPERIMENT_2_NAME", "mlflow-agent-2")

PROMPT_NAME = "mlflow-agent-system"
PROMPT_ALIAS = f"prompts:/{PROMPT_NAME}@prod"


def mlflow_agent(question: str) -> str:
    """MLflow assistant with updated prompt."""
    system_prompt = mlflow.genai.load_prompt("prompts:/mlflow-agent-system@prod")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt.template},
            {"role": "user",   "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

def predict_fn(question: str) -> str:
    return mlflow_agent(question)

2026/04/25 10:29:08 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.


## Problem Validation

In [ ]:
retrieval_eval_dataset = [
    # ─────────────────────────────────────────────────────────────────────────
    # Q1: Specific named optimizers for judge alignment
    # Why retrieval-only: SIMBA and GEPA are highly specific names. A bare LLM
    # will typically say "use RLHF" or "fine-tune the judge" — generic ML advice.
    # Only doc5 names these optimizers and the .align() workflow.
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "How do I align an LLM judge in MLflow to match human "
                        "evaluation standards, and which optimizers are supported?"
        },
        "expectations": {
            "expected_facts": ["make_judge", "align", "SIMBA", "GEPA", "log_feedback"],
        },
    },

    # ─────────────────────────────────────────────────────────────────────────
    # Q2: Built-in scorer names + custom scorer construction
    # Why retrieval-only: A bare LLM tends to invent plausible-but-wrong scorer
    # names ("Accuracy", "Faithfulness", "Helpfulness"). Only doc3 lists the
    # actual MLflow 3.0 names AND distinguishes Guidelines() vs make_judge().
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "What built-in scorers does MLflow provide for GenAI "
                        "evaluation, and what are the two ways to create custom scorers?"
        },
        "expectations": {
            "expected_facts": [
                "Correctness", "Safety", "RelevanceToQuery", "RetrievalGroundedness",
                "Guidelines", "make_judge",
            ],
        },
    },

    # ─────────────────────────────────────────────────────────────────────────
    # Q3: Prompt Registry API specifics
    # Why retrieval-only: The exact function names register_prompt / load_prompt
    # under mlflow.genai.* are post-cutoff API. Bare LLMs guess "mlflow.log_prompt"
    # or describe Langfuse/PromptLayer patterns. Only doc4 has the real API.
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "How do I version and retrieve prompts in MLflow, and "
                        "what lifecycle stages do prompt aliases support?"
        },
        "expectations": {
            "expected_facts": [
                "mlflow.genai.register_prompt", "mlflow.genai.load_prompt",
                "development", "staging", "production",
            ],
        },
    },

    # ─────────────────────────────────────────────────────────────────────────
    # Q4: AI Gateway capabilities — multiple specific features
    # Why retrieval-only: "AI Gateway" is the rebranded name for what used to be
    # "MLflow Deployments for LLMs". Bare LLMs either confuse it with the older
    # name or invent generic gateway features. Doc2 has the canonical feature list.
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "What does the MLflow AI Gateway do, and what features "
                        "does it provide for managing multiple LLM providers?"
        },
        "expectations": {
            "expected_facts": [
                "single", "endpoint", "rate limiting", "fallback",
                "budget"
            ],
        },
    },

    # ─────────────────────────────────────────────────────────────────────────
    # Q5: Tracing API surface — decorator + context manager
    # Why retrieval-only: This is your signature Stage 1 failure. Pre-3.0 training
    # data points models toward `mlflow.start_run()` and they invent decorators
    # like `@mlflow.log_trace`. Only doc1 has both @mlflow.trace AND
    # mlflow.start_span() together with the autologging path.
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "What are the three ways to capture traces for a GenAI "
                        "application in MLflow?"
        },
        "expectations": {
            "expected_facts": [
                "autologging", "@mlflow.trace", "mlflow.start_span",
            ],
        },
    },
    # ─────────────────────────────────────────────────────────────────────────
    # Q6: Cross-document synthesis (cost tracking + tracing + search)
    # Why retrieval-only: Requires combining doc8 (cost tracking, search_traces)
    # with doc1 (spans/autologging context). A bare LLM will describe generic
    # "log token counts to a CSV" patterns. The specific function name
    # mlflow.search_traces() is the retrieval tell.
    # ─────────────────────────────────────────────────────────────────────────
    {
        "inputs": {
            "question": "Does MLflow support the deployment of local inference servers?"
        },
        "expectations": {
            "expected_facts": [
                "yes"
            ],
        },
    },
    {
        "inputs": {"question": "How do I run an LLM or agent evaluation in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.genai.evaluate", "data", "scorers"],
            "guidelines": [
                "Response must include a concrete code example, not just a description of what to do",
                "Response must reference specific parameter names, not just say 'pass in your data'",
            ],
        },
    },
    # Version clarity — requires distinguishing 3.0 from pre-3.0
    {
        "inputs": {"question": "How do I add a custom scorer to my evaluation?"},
        "expectations": {
            "expected_facts": ["@scorer", "mlflow.genai"],
            "guidelines": [
                "Response must indicate this API is specific to MLflow 3.0",
                "Response must not reference mlflow.evaluate() which is the pre-3.0 pattern",
            ],
        },
    },
    # Tone — conversational vs. changelog
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {
            "guidelines": [
                "Response should be conversational and helpful, not a dry enumeration of features",
                "Response should explain the practical benefit to the user, not just what the feature does",
            ],
        },
    },
    # Combines all three — specificity, version clarity, and tone
    {
        "inputs": {"question": "How do I trace a custom function in MLflow 3.0?"},
        "expectations": {
            "expected_facts": ["@mlflow.trace", "SpanType"],
            "guidelines": [
                "Response must include a code example showing the decorator in use",
                "Response must clarify this is a MLflow 3.0 feature",
                "Response should be conversational and explain when you would want to do this",
            ],
        },
    },
]

In [25]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery
from mlflow.genai import evaluate

first_eval_results = evaluate(
    data=retrieval_eval_dataset,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(model=f"gemini:/{MODEL}"),
        RelevanceToQuery(model=f"gemini:/{MODEL}"),
        ],
)

print(first_eval_results.metrics)

2026/04/25 10:29:10 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 6/6 [Elapsed: 00:04, Remaining: 00:00] [predict_fn: 75%, scorers: 25%]


{'relevance_to_query/mean': np.float64(1.0), 'correctness/mean': np.float64(0.16666666666666666)}


---
## 1 — The Document Store

In production this would be a vector database. For this workshop we use a plain dictionary — 10 documents covering core MLflow 3 capabilities that our assistant should be able to reference.

In [4]:
DOCUMENT_STORE = {
    "doc1": (
        "MLflow Tracing provides end-to-end observability for GenAI applications. "
        "It captures every LLM call, retrieval step, tool invocation, and agent reasoning "
        "as structured spans with full input/output visibility. Traces are logged automatically "
        "via autologging or manually with the @mlflow.trace decorator and mlflow.start_span() context manager."
    ),
    "doc2": (
        "MLflow AI Gateway lets teams manage multiple LLM providers through a single, secure endpoint. "
        "It centralizes access control, cost tracking, and rate limiting across providers like OpenAI, "
        "Anthropic, and Google. Features include traffic routing, automatic fallbacks between providers, "
        "budget alerts, and comprehensive usage analytics."
    ),
    "doc3": (
        "MLflow Evaluation enables systematic testing of GenAI applications using scorers. "
        "Built-in scorers include Correctness, Safety, RelevanceToQuery, and RetrievalGroundedness. "
        "Teams can also create custom scorers with Guidelines() for natural-language criteria "
        "or make_judge() for LLM-as-judge evaluators. Run evaluations with mlflow.genai.evaluate()."
    ),
    "doc4": (
        "MLflow Prompt Registry allows teams to version, share, and manage prompts centrally. "
        "Prompts support template variables, lifecycle aliases (development, staging, production), "
        "and integration with experiments. Use mlflow.genai.register_prompt() to store prompts "
        "and mlflow.genai.load_prompt() to retrieve them by name and version."
    ),
    "doc5": (
        "Judge alignment in MLflow teaches LLM judges to match human evaluation standards. "
        "The workflow is: create a judge with make_judge(), collect human feedback on traces "
        "via mlflow.log_feedback(), then call judge.align() with the SIMBA or GEPA optimizer. "
        "Aligned judges typically show 30-50% reduction in false positives compared to generic prompts."
    ),
    "doc6": (
        "MLflow autologging automatically captures metrics, parameters, and traces for 40+ frameworks "
        "including OpenAI, LangChain, LlamaIndex, DSPy, and AutoGen. Enable it with one line — "
        "e.g., mlflow.openai.autolog() or mlflow.langchain.autolog(). Each call is traced with "
        "token counts, latency, and the full request/response payload."
    ),
    "doc7": (
        "MLflow Deployments serves models as production-ready REST API endpoints. "
        "It supports local inference servers, Kubernetes, AWS SageMaker, and managed platforms. "
        "Models are deployed directly from the Model Registry with version pinning, "
        "and each serving endpoint gets automatic request/response logging."
    ),
    "doc8": (
        "MLflow tracks token usage and cost across LLM applications. Every traced call records "
        "input tokens, output tokens, and total tokens. When provider pricing is configured, "
        "MLflow calculates per-call and cumulative costs. Use mlflow.search_traces() to query "
        "usage patterns and identify expensive operations."
    ),
    "doc9": (
        "MLflow Experiment Tracking organizes ML and GenAI work into experiments and runs. "
        "Each run logs parameters, metrics, artifacts, and traces. Teams use mlflow.set_experiment() "
        "to group related work, mlflow.log_param() and mlflow.log_metric() for tracking, "
        "and the MLflow UI to compare results across runs visually."
    ),
    "doc10": (
        "MLflow is fully open source and vendor-neutral. It works with any cloud provider, "
        "ML framework, or LLM provider without lock-in. The unified API covers the full lifecycle "
        "from experimentation through evaluation to production deployment, with a single tracking "
        "server that stores all artifacts, metrics, and traces."
    ),
}

print(f"Document store initialized with {len(DOCUMENT_STORE)} documents")
for doc_id, text in DOCUMENT_STORE.items():
    print(f"  {doc_id}: {text[:80]}...")

Document store initialized with 10 documents
  doc1: MLflow Tracing provides end-to-end observability for GenAI applications. It capt...
  doc2: MLflow AI Gateway lets teams manage multiple LLM providers through a single, sec...
  doc3: MLflow Evaluation enables systematic testing of GenAI applications using scorers...
  doc4: MLflow Prompt Registry allows teams to version, share, and manage prompts centra...
  doc5: Judge alignment in MLflow teaches LLM judges to match human evaluation standards...
  doc6: MLflow autologging automatically captures metrics, parameters, and traces for 40...
  doc7: MLflow Deployments serves models as production-ready REST API endpoints. It supp...
  doc8: MLflow tracks token usage and cost across LLM applications. Every traced call re...
  doc9: MLflow Experiment Tracking organizes ML and GenAI work into experiments and runs...
  doc10: MLflow is fully open source and vendor-neutral. It works with any cloud provider...


---
## 2 — Building the RAG Pipeline (without tracing)

A RAG pipeline has four stages:

| Stage | Function | What it does |
|---|---|---|
| **Embed** | `get_embeddings()` | Turn text into vectors |
| **Index** | `build_index()` | Embed all documents up front |
| **Search** | `semantic_search()` | Find the most relevant docs for a query |
| **Generate** | `rag_query()` | Build a grounded prompt and call the LLM |

We'll build each one as a plain Python function first.

### 2a — Embedding function

We use the Gemini embeddings API through the OpenAI-compatible endpoint. This returns a dense vector for each input text.

In [6]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts using the Gemini embedding model."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

# Quick test
test_emb = get_embeddings(["hello world"])
print(f"Embedding dim: {len(test_emb[0])}")

Embedding dim: 3072


Trace(trace_id=tr-dcf0b7cc776b2ec500558c171d9f52eb)

### 2b — Build the document index

We embed every document once and store the vectors alongside the text. In production you'd use a vector DB — here a list of dicts is fine.

In [7]:
def build_index(document_store: dict[str, str]) -> list[dict]:
    """Embed all documents and return an index of {id, text, embedding}."""
    doc_ids = list(document_store.keys())
    doc_texts = list(document_store.values())
    embeddings = get_embeddings(doc_texts)

    index = []
    for doc_id, text, emb in zip(doc_ids, doc_texts, embeddings):
        index.append({"id": doc_id, "text": text, "embedding": emb})

    return index

doc_index = build_index(DOCUMENT_STORE)
print(f"Indexed {len(doc_index)} documents")

Indexed 10 documents


Trace(trace_id=tr-37f6b5c1723ebc987906c9c15185b01f)

### 2c — Semantic search

Given a query, embed it and find the `top_k` most similar documents using cosine similarity.

In [8]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def semantic_search(query: str, index: list[dict], top_k: int = 3) -> list[dict]:
    """Find the top_k most relevant documents for a query."""
    query_embedding = get_embeddings([query])[0]

    scored = []
    for doc in index:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({"id": doc["id"], "text": doc["text"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

# Test it
results = semantic_search("How do I trace my LLM calls?", doc_index)
for r in results:
    print(f"  [{r['score']:.3f}] {r['id']}: {r['text'][:80]}...")

  [0.720] doc8: MLflow tracks token usage and cost across LLM applications. Every traced call re...
  [0.672] doc1: MLflow Tracing provides end-to-end observability for GenAI applications. It capt...
  [0.636] doc6: MLflow autologging automatically captures metrics, parameters, and traces for 40...


Trace(trace_id=tr-98a2b0f026215599aa302eba104ec59b)

### 2d — Prompt construction

This is the "augmented" part of RAG — we inject the retrieved documents into the prompt so the LLM can ground its answer in real content.

In [9]:
SYSTEM_PROMPT = """You are a helpful MLflow assistant. Answer the user's question using ONLY
the provided context documents. If the context doesn't contain enough information
to answer fully, say so honestly.

Context:
{context}"""


def build_prompt(question: str, retrieved_docs: list[dict]) -> list[dict]:
    """Construct the chat messages with retrieved context injected."""
    context = "\n\n".join(
        f"[{doc['id']}] {doc['text']}" for doc in retrieved_docs
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ]

# Preview what the prompt looks like
sample_messages = build_prompt("How do I trace my LLM calls?", results)
print("System prompt (first 300 chars):")
print(sample_messages[0]["content"][:300])
print(f"\nUser: {sample_messages[1]['content']}")

System prompt (first 300 chars):
You are a helpful MLflow assistant. Answer the user's question using ONLY
the provided context documents. If the context doesn't contain enough information
to answer fully, say so honestly.

Context:
[doc8] MLflow tracks token usage and cost across LLM applications. Every traced call records input t

User: How do I trace my LLM calls?


### 2e — The complete RAG query function

Ties it all together: search -> build prompt -> call LLM.

In [10]:
def rag_query(question: str, index: list[dict], top_k: int = 3) -> str:
    """End-to-end RAG: retrieve relevant docs, build prompt, generate answer."""
    # Retrieve
    retrieved_docs = semantic_search(question, index, top_k=top_k)

    # Augment
    messages = build_prompt(question, retrieved_docs)

    # Generate
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    return response.choices[0].message.content

In [11]:
# Try it out
answer = rag_query("How do I trace my LLM calls in MLflow?", doc_index)
print(answer)

You can trace your LLM calls in MLflow using the following methods:

*   **Autologging:** MLflow autologging automatically captures traces for various frameworks like OpenAI, LangChain, LlamaIndex, DSPy, and AutoGen. You can enable it with a single line of code, for example, `mlflow.openai.autolog()` or `mlflow.langchain.autolog()`. Each call will be traced with token counts, latency, and the full request/response payload.
*   **Manual Tracing:** You can manually trace calls using the `@mlflow.trace` decorator or the `mlflow.start_span()` context manager.


[Trace(trace_id=tr-6cd1542a8ff67dfb3c0c22f03fa64784), Trace(trace_id=tr-3ca00f5fc054bb3769c23bf4711bd5cc)]

In [12]:
answer = rag_query("What is the AI Gateway and what does it do?", doc_index)
print(answer)

MLflow AI Gateway allows teams to manage multiple LLM providers through a single, secure endpoint. It centralizes access control, cost tracking, and rate limiting across providers such as OpenAI, Anthropic, and Google. Its features include traffic routing, automatic fallbacks between providers, budget alerts, and detailed usage analytics.


[Trace(trace_id=tr-be457e2a7cc02ae46748d3878abeb9ae), Trace(trace_id=tr-c89b5d235e90ff0409b88014670111c0)]

---
## 3 — The problem: it works, but it's a black box

The pipeline returns answers, but we can't see *inside* it:

- Which documents were retrieved? Were they relevant?
- How long did embedding take vs. generation?
- What exact prompt was sent to the LLM?
- How many tokens did it use?

Without observability, debugging is guesswork. Let's fix that with **MLflow Tracing**.

---
## 4 — Adding MLflow Tracing

We'll re-define the same functions, but now with tracing decorators. MLflow gives us two tools:

| Tool | Use when |
|---|---|
| `@mlflow.trace` | Decorating a function — captures inputs, outputs, and timing automatically |
| `mlflow.start_span()` | You need a span *inside* a function (e.g., around a loop or API call) |

Each span gets a `span_type` that tells the MLflow UI how to render it:

| Span type | Meaning |
|---|---|
| `EMBEDDING` | A vectorization call |
| `RETRIEVER` | A document lookup/search |
| `LLM` | A language model call |
| `CHAIN` | A multi-step orchestration |

In [13]:
from mlflow.entities import SpanType

### 4a — Traced embedding function

The `@mlflow.trace` decorator wraps the function in a span. We set `span_type=SpanType.EMBEDDING` so the UI knows this is a vectorization step.

In [14]:
@mlflow.trace(span_type=SpanType.EMBEDDING)
def get_embeddings_traced(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts (traced)."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]

### 4b — Traced semantic search

The retriever span captures the query and the documents it found — this is critical for debugging relevance issues.

In [15]:
@mlflow.trace(span_type=SpanType.RETRIEVER)
def semantic_search_traced(query: str, index: list[dict], top_k: int = 3) -> list[dict]:
    """Find the top_k most relevant documents for a query (traced)."""
    query_embedding = get_embeddings_traced([query])[0]

    scored = []
    for doc in index:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({"id": doc["id"], "text": doc["text"], "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

### 4c — Traced prompt construction

In [16]:
@mlflow.trace(span_type=SpanType.CHAIN)
def build_prompt_traced(question: str, retrieved_docs: list[dict]) -> list[dict]:
    """Construct the chat messages with retrieved context injected (traced)."""
    context = "\n\n".join(
        f"[{doc['id']}] {doc['text']}" for doc in retrieved_docs
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ]

### 4d — Traced RAG query

The top-level function uses `span_type=SpanType.CHAIN` to group the entire pipeline into one trace. Every sub-function creates a child span automatically.

In [17]:
@mlflow.trace(span_type=SpanType.CHAIN)
def rag_query_traced(question: str, index: list[dict], top_k: int = 3) -> str:
    """End-to-end RAG with full MLflow tracing."""
    # Retrieve — creates a RETRIEVER span
    retrieved_docs = semantic_search_traced(question, index, top_k=top_k)

    # Augment — creates a CHAIN span
    messages = build_prompt_traced(question, retrieved_docs)

    # Generate — creates an LLM span via start_span
    with mlflow.start_span(name="llm_generate", span_type=SpanType.LLM) as span:
        span.set_inputs({"messages": messages, "model": MODEL})
        response = openai_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.2,
        )
        answer = response.choices[0].message.content
        span.set_outputs({"answer": answer})
        span.set_attributes({
            "token_count.input": response.usage.prompt_tokens,
            "token_count.output": response.usage.completion_tokens,
        })

    return answer

---
## 5 — Run the traced pipeline

Same questions as before — but now every step is visible in the MLflow UI. Open the **Traces** tab to see the full span tree.

In [18]:
test_questions = [
    "How do I trace my LLM calls in MLflow?",
    "What is the AI Gateway and what does it do?",
    "How do I evaluate my agent's responses?",
    "What is judge alignment and how does it work?",
    "How do I version my prompts in MLflow?",
]

for q in test_questions:
    print(f"Q: {q}")
    answer = rag_query_traced(q, doc_index)
    print(f"A: {answer[:200]}...\n")

Q: How do I trace my LLM calls in MLflow?
A: You can trace your LLM calls in MLflow using the following methods:

*   **Autologging**: MLflow autologging automatically captures traces for various frameworks like OpenAI, LangChain, LlamaIndex, DS...

Q: What is the AI Gateway and what does it do?
A: MLflow AI Gateway allows teams to manage multiple LLM providers through a single, secure endpoint. It centralizes access control, cost tracking, and rate limiting across providers such as OpenAI, Anth...

Q: How do I evaluate my agent's responses?
A: You can evaluate your agent's responses using MLflow Evaluation. This feature allows for systematic testing of GenAI applications using scorers. You can use built-in scorers like Correctness, Safety, ...

Q: What is judge alignment and how does it work?
A: Judge alignment in MLflow is a process that teaches LLM judges to match human evaluation standards. It works through a workflow that involves:

1.  **Creating a judge:** This is done using the

[Trace(trace_id=tr-51d26c16f69b1669e67d91476764d040), Trace(trace_id=tr-a5b991b855cad300bf62affa422e981f), Trace(trace_id=tr-c2b69f80837b578fb4ae5cfab19a843a), Trace(trace_id=tr-adeecd5cab1654ad4586ddb3a0a9b431), Trace(trace_id=tr-34577b8705e8cdf1b9c29374920392f2)]

---
## 6 — What tracing reveals

Open the MLflow UI (`http://127.0.0.1:5000`) and click on any trace. You should see:

```
rag_query_traced (CHAIN)
  |-- semantic_search_traced (RETRIEVER)
  |     |-- get_embeddings_traced (EMBEDDING)
  |-- build_prompt_traced (CHAIN)
  |-- llm_generate (LLM)
```

For each span you can inspect:
- **Inputs/Outputs** — what went in and what came out
- **Latency** — how long each step took
- **Token counts** — on the LLM span
- **Retrieved docs** — on the RETRIEVER span, so you can check relevance

This is the foundation for debugging and evaluation. In the next notebook, we'll use these traces to run scorers like `RetrievalGroundedness` that check whether the answer is actually supported by the retrieved context.

## Check Improvement

In [19]:
def mlflow_agent(question: str) -> str:
    """MLflow assistant with retrieval augmentation and tracing."""
    return rag_query_traced(question, doc_index)

def predict_fn(question: str) -> str:
    return mlflow_agent(question)

In [20]:
first_eval_results = evaluate(
    data=retrieval_eval_dataset,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(model=f"gemini:/{MODEL}"),
        RelevanceToQuery(model=f"gemini:/{MODEL}"),
        ],
)

print(first_eval_results.metrics)

2026/04/25 10:23:43 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 6/6 [Elapsed: 00:15, Remaining: 00:00] [predict_fn: 81%, scorers: 19%]


{'relevance_to_query/mean': np.float64(1.0), 'correctness/mean': np.float64(0.8333333333333334)}


In [ ]:
#Add examples to evaluation dataset